# Problem Statement
1. Intelligent Pricing & Discount Optimization
2. Recommends dynamic pricing based on customer segments, historical data, and market competition.
3. Suggests optimal discount levels to maximize margins without losing deals.

# Strategy Followed
    ┌──────────────────────┐    ┌───────────────────┐    ┌──────────────────────────┐    ┌────────────────────┐    ┌───────────────────┐
    │   Synthetic Data     │    │  Data Prep Layer  │    │  ML Modeling Layer       │    │ Optimization Layer │    │  Results / UI     │
    │ (customers, prices,  │--->│  - clean data     │--->│ 1) Price Elasticity      │--->│ Max profit / deal  │--->│ Print tables,     │
    │  discounts, sales)   │    │  - feature engine │    │ 2) Deal Conversion Model │    │ Suggest price      │    │ graphs, insights  │
    └──────────────────────┘    └───────────────────┘    └──────────────────────────┘    └────────────────────┘    └───────────────────┘
                   

In [1]:
!pip install xgboost
!pip install bayesian-optimization

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from xgboost import XGBClassifier
from bayes_opt import BayesianOptimization

In [2]:
# note : change the path to your local system
# data = pd.read_csv('pricing_optimization_synthetic_data.csv')
GITHUB_RAW_URL_FOR_FILE = "https://raw.githubusercontent.com/rahilkhan-acadmic/APAIML-GradedMiniProject/refs/heads/develop/AIChampion/pricing_optimization_synthetic_data.csv"
data = pd.read_csv(GITHUB_RAW_URL_FOR_FILE)

FileNotFoundError: [Errno 2] No such file or directory: 'pricing_optimization_synthetic_data.csv'

In [ ]:
data

In [ ]:
print(f'data.shape : {data.shape}')
print(f'\n\n-------------- data.describe() -------------- \n{data.describe()}')
print(f'\n\n-------------- data.info() --------------  ')
data.info()

## Step 3. Feature Engineering
### Alternative: Ordinal Encoding for Categorical Features

Ordinal encoding assigns a unique integer to each category. This can be suitable when there's an inherent order in your categories, or if your model (like tree-based models) can interpret these numerical representations without assuming an artificial order.

In [ ]:
data_ordinal = data.copy()

# Drop the 'customer_id' column
data_ordinal.drop(columns=['customer_id'], axis=1, inplace=True)

# Apply Ordinal Encoding to 'customer_segment' and 'market_condition'
ordinal_encoder = OrdinalEncoder()
data_ordinal[['customer_segment', 'market_condition']] = ordinal_encoder.fit_transform(data_ordinal[['customer_segment', 'market_condition']])

# Display the head of the new DataFrame with ordinal encoded columns
display(data_ordinal.head())

### Subtask:
Create a binary target variable 'deal_converted' in the `data_ordinal` DataFrame, setting it to 1 if `optimal_discount_%` is positive and 0 otherwise.

In [ ]:
data_ordinal['deal_converted'] = (data_ordinal['optimal_discount_%'] > 0).astype(int)
display(data_ordinal.head())

## Build Price Elasticity Model

### Subtask:
Implement a Log-linear regression model to predict price elasticity using `demand_score` as the target variable and `historical_price` as a key feature, along with other relevant features from `data_ordinal`.

**Reasoning:** This subtask requires implementing a Log-linear regression model. The first step is to define the features (X) and target (y) from the `data_ordinal` DataFrame as specified, apply logarithmic transformations to `historical_price` and `demand_score`, and then split the data into training and testing sets, before initializing and training a Linear Regression model.

In [ ]:
X = data_ordinal[['historical_price', 'competitor_price', 'purchase_frequency_last_6m', 'customer_segment', 'market_condition']]
y = data_ordinal[['demand_score']]


# Log operation is applied to achieve following objective
# 1. Linearizing Relationships:
#     Relationship b/w price & demand is non-linear(e.g. exponential or multiplicative).
#     Apply logarithm -> transform - non-linear relationship ->linear
# 2. Model Specification:
#     In a log-log model where both price and demand are log-transformed, the model's coefficient for price directly reveals
#     how much demand changes percentage-wise for each one percent change in price, which is known as price elasticity.
# 3. Normalizing Skewed Data:
#     Price data -> distribution like data. Logarithmic transformations -> make distribution - symmetric & close -normal distribution.
#     Improve performance and validity of linear regression models.
X_elasticity = X.copy()
X_elasticity['historical_price'] = np.log(X_elasticity['historical_price'])
y_elasticity = np.log(y)

print(f'X_elasticity: \n{X_elasticity}')
print(f'y_elasticity: \n{y_elasticity}')

X_train_elasticity, X_test_elasticity, y_train_elasticity, y_test_elasticity = train_test_split(X_elasticity, y_elasticity, test_size=0.3, random_state=101)
log_linear_model = LinearRegression()
log_linear_model.fit(X_train_elasticity, y_train_elasticity)

print("Log-linear regression model trained successfully.")

## Evaluate Models and Visualize Results

### Subtask:
Evaluation of performance of the Log-linear regression model using following metrics:
1. Mean Squared Error (MSE)
2. Mean Absolute Error (MBE)
3. R-squared score (R2)

In [ ]:
#1. Predict the demand score for the test set
y_pred_elasticity = log_linear_model.predict(X_test_elasticity)

#2. Calculate MSE
mse_elasticity = mean_squared_error(y_test_elasticity, y_pred_elasticity)

#3. Calculate MAE
mae_elasticity = mean_absolute_error(y_test_elasticity, y_pred_elasticity)

#4. Calculate R2 score
r2_elasticity = r2_score(y_test_elasticity, y_pred_elasticity)

print(f"Log-linear regression Model Performance:")
print(f"Mean Squared Error (MSE): {mse_elasticity:.4f}")
print(f"Mean Absolute Error (MAE): {mae_elasticity:.4f}")
print(f"R-squared (R2) Score: {r2_elasticity:.4f}")

## Build Deal Conversion Model

### Subtask:
Develop an XGBoost Classifier to predict deal conversion.
The model will use the engineered features from the data preparation step and the newly created binary target variable `deal_converted`.

**Reasoning**:
To develop the XGBoost Classifier, I will first define the feature matrix `X_conversion` and the target variable `y_conversion` as specified.
Then, I will split the data into training and testing sets using `train_test_split`.
Finally, I will initialize and train the `XGBClassifier` model.



In [ ]:
# 1. Define the feature matrix X_conversion and target variable y_conversion
X_conversion = data_ordinal[['historical_price', 'competitor_price', 'purchase_frequency_last_6m', 'customer_segment', 'market_condition']]
y_conversion = data_ordinal['deal_converted']

#2. Split the data into training and test sets
X_train_conversion, X_test_conversion, y_train_conversion, y_test_conversion = train_test_split(X_conversion, y_conversion, test_size=0.3, random_state=101)

#3. Initialize an XGBClassifier model
xgb_model = XGBClassifier(eval_metric='logloss', random_state=101)

#4. Train the XGBClassifier model
xgb_model.fit(X_train_conversion, y_train_conversion)

print("XGBoost Classifier trained successfully")

## Evaluate Models and Visualize Results

### Subtask:
Evaluate the performance of the XGBoost Classifier model using appropriate metrics.

**Reasoning**:
To evaluate the XGBoost Classifier, I will first predict probabilities and class labels for the test set. Then, I will calculate and print the accuracy score, ROC AUC score, and a classification report.



In [ ]:
# 1. Predict the deal_converted probabilities for the test set
y_pred_proba_conversion = xgb_model.predict_proba(X_test_conversion)[:, 1]

# 2. Predict the deal_converted labels for the test set
y_pred_conversion = xgb_model.predict(X_test_conversion)

# 3. Calculate the Accuracy Score
accuracy_conversion = accuracy_score(y_test_conversion, y_pred_conversion)

# 4. Calculate the Area Under the Receiver Operating Characteristic Curve (ROC AUC Score)
roc_auc_conversion = roc_auc_score(y_test_conversion, y_pred_proba_conversion)

# 5. Generate a classification report
classification_rep_conversion = classification_report(y_test_conversion, y_pred_conversion)

# 6. Print all the calculated metrics
print(f"XGBoost Classifier Model Performance:")
print(f"Accuracy: {accuracy_conversion:.4f}")
print(f"ROC AUC Score: {roc_auc_conversion:.4f}")
print(f"Classification Report:\n{classification_rep_conversion}")

**Precision:** Proportion of positive predictions that are actually correct:
*   `Precision = TP/(TP +FP)`
    
**Recal/Sensitivity:** Proportion of actual positive predictions that are correctly identified
*   `Recal = TP/(TP+FN)`

**F1-Score:** Harmonic Mean of `Precision` and `Recal`
*   `Recal = 2*(Precision * Recal)/(Precision + Recal)`

**Support:** No. of actual occurances of each class in the dataset. It provides context for other metrics but does not directly measures model performance.

**Accuracy:** Proportion of all predictions(both +ve and -ve) that are correct.
*Caveat: Misleading for imbalanced dataset. A model predicting only the majority class achieves high accuracy but fails to identify the minority class.*
*   `Accuracy = (TP + TN)/Total Predictions`        

    
    

## Implement Pricing Optimization

### Subtask:
Set up a framework for pricing optimization using Bayesian Optimization. This step will involve defining an objective function that leverages the predictions from both the Price Elasticity and Deal Conversion models to find optimal pricing strategies.


In [ ]:
#1. Caluclate the median values for relevant features from data_ordinal
median_competitor_price = data_ordinal['competitor_price'].median()
median_purchase_frequency = data_ordinal['purchase_frequency_last_6m'].median()
median_customer_segment = data_ordinal['customer_segment'].median()
median_market_condition = data_ordinal['market_condition'].median()

#2. Define the objective _fn() for Bayesian optimization
def objective_function(price_val):
    if price_val <= 0:
        return -np.inf #This ensure +ve value is returned

    # Prepare fts for the log_linear_model(price elasticity model)
    # X_elasticity has columns: 'historic_price', 'competitor_price', 'purchase_frequenc_last_6m','customer_segment', 'market_condition'
    features_elasticity = pd.DataFrame({
        'historical_price': [np.log(price_val)],
        'competitor_price': [median_competitor_price],
        'purchase_frequency_last_6m': [median_purchase_frequency],
        'customer_segment': [median_customer_segment],
        'market_condition': [median_market_condition]
    })

    # Ensure column order matches the training data for the model
    features_elasticity = features_elasticity[X_elasticity.columns]

    # Predict log_demand_score and then exponentiate to get predicted demand score
    # Corrected to ensure log_demand_score is a scalar
    log_demand_score = log_linear_model.predict(features_elasticity)[0][0]
    predicted_demand_score = np.exp(log_demand_score)

    # Prepare features for the xgb_model (deal conversion model)
    # X_conversion has columns: 'historical_price', 'competitor_price', 'purchase_frequency_last_6m', 'customer_segment', 'market_condition'
    features_conversion = pd.DataFrame({
        'historical_price': [price_val],
        'competitor_price': [median_competitor_price],
        'purchase_frequency_last_6m': [median_purchase_frequency],
        'customer_segment': [median_customer_segment],
        'market_condition': [median_market_condition]
    })

    # Ensure column order matches the training data for the model
    features_conversion = features_conversion[X_conversion.columns]

    # Predict conversion_probability
    conversion_probability = xgb_model.predict_proba(features_conversion)[:, 1][0]

    # Calculate expected revenue
    expected_revenue = price_val * predicted_demand_score * conversion_probability
    return expected_revenue

# 4. Define the bounds for price_val
price_bounds = {
    'price_val': (data_ordinal['historical_price'].min(), data_ordinal['historical_price'].max())
}

# 5. Initialize BayesianOptimization
optimizer = BayesianOptimization(
    f=objective_function,
    pbounds=price_bounds,
    random_state=101,
)

# 6. Run the Bayesian Optimization
optimizer.maximize(
    init_points=20,  # Number of steps of random exploration
    n_iter=50,       # Number of steps of Bayesian optimization
)

# Print the best parameters and the corresponding maximum value
print("\nOptimal Pricing Strategy:")
print(f"Best expected revenue: {optimizer.max['target']:.2f}")
print(f"Optimal price: {optimizer.max['params']['price_val']:.2f}")


## Visualize Results and Feature Importance

### Subtask:
Visualize key findings, feature importances, and the outcomes of the pricing optimization to derive actionable insights.

In [ ]:
import matplotlib.pyplot as plt

# Extract feature importances from the XGBoost model
feature_importances = xgb_model.feature_importances_
feature_names = X_conversion.columns

# Create a DataFrame for easier plotting
d = pd.DataFrame({'Feat_name': feature_names, 'Feat_imp': feature_importances})

# Sort by importance for better visualization
d = d.sort_values(by='Feat_imp', ascending=True)

# 1. Visualize the feature importances for the XGBoost model
plt.figure(figsize=(10, 6))
plt.barh(d['Feat_name'], d['Feat_imp'])
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('XGBoost Feature Importance for Deal Conversion Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
#2. Extract the results from the optimizer
optimization_results = pd.DataFrame(optimizer.res)

#3. Plot the expected revenue against the explored price_val values
plt.figure(figsize=(10,6))
plt.plot(optimization_results['params'].apply(lambda x: x['price_val']), optimization_results['target'], marker='o', linestyle='-')
plt.xlabel('Price Value')
plt.ylabel('Expected Revenue')
plt.title('Bayesian Optimization: Expected Revenue vs. Price')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#Updated approach:
def objective_function(price_val):
    if price_val <= 0:
        return -np.inf

    # DEBUG: Print the price being tested
    print(f"\nTesting price: ${price_val:.2f}")

    # Prepare features for elasticity model
    features_elasticity = pd.DataFrame({
        'historical_price': [np.log(price_val)],
        'competitor_price': [median_competitor_price],
        'purchase_frequency_last_6m': [median_purchase_frequency],
        'customer_segment': [median_customer_segment],
        'market_condition': [median_market_condition]
    })
    features_elasticity = features_elasticity[X_elasticity.columns]

    # Get demand prediction
    log_demand_score = log_linear_model.predict(features_elasticity)[0][0]
    predicted_demand_score = np.exp(log_demand_score)

    # DEBUG: Show demand prediction
    print(f"Log demand score: {log_demand_score:.4f}")
    print(f"Predicted demand score: {predicted_demand_score:.4f}")

    # Prepare features for conversion model
    features_conversion = pd.DataFrame({
        'historical_price': [price_val],
        'competitor_price': [median_competitor_price],
        'purchase_frequency_last_6m': [median_purchase_frequency],
        'customer_segment': [median_customer_segment],
        'market_condition': [median_market_condition]
    })
    features_conversion = features_conversion[X_conversion.columns]

    # Get conversion probability
    conversion_probability = xgb_model.predict_proba(features_conversion)[:, 1][0]

    # DEBUG: Show conversion probability
    print(f"Conversion probability: {conversion_probability:.4f}")

    # Calculate expected revenue
    expected_revenue = price_val * predicted_demand_score * conversion_probability

    # DEBUG: Show calculation
    print(f"Expected revenue: ${price_val:.2f} × {predicted_demand_score:.4f} × {conversion_probability:.4f} = ${expected_revenue:.2f}")

    return expected_revenue

# 4. Define the bounds for price_val
price_bounds = {
    'price_val': (data_ordinal['historical_price'].min(), data_ordinal['historical_price'].max())
}

# 5. Initialize BayesianOptimization
optimizer = BayesianOptimization(
    f=objective_function,
    pbounds=price_bounds,
    random_state=101,
)

# 6. Run the Bayesian Optimization
optimizer.maximize(
    init_points=20,  # Number of steps of random exploration
    n_iter=50,       # Number of steps of Bayesian optimization
)

# Print the best parameters and the corresponding maximum value
print("\nOptimal Pricing Strategy:")
print(f"Best expected revenue: {optimizer.max['target']:.2f}")
print(f"Optimal price: {optimizer.max['params']['price_val']:.2f}")

In [ ]:

# Extract feature importances from the XGBoost model
feature_importances = xgb_model.feature_importances_
feature_names = X_conversion.columns

# Create a DataFrame for easier plotting
d = pd.DataFrame({'Feat_name': feature_names, 'Feat_imp': feature_importances})

# Sort by importance for better visualization
d = d.sort_values(by='Feat_imp', ascending=True)

# 1. Visualize the feature importances for the XGBoost model
plt.figure(figsize=(10, 6))
plt.barh(d['Feat_name'], d['Feat_imp'])
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('XGBoost Feature Importance for Deal Conversion Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
#2. Extract the results from the optimizer
optimization_results = pd.DataFrame(optimizer.res)

#3. Plot the expected revenue against the explored price_val values
plt.figure(figsize=(10,6))
plt.plot(optimization_results['params'].apply(lambda x: x['price_val']), optimization_results['target'], marker='o', linestyle='-')
plt.xlabel('Price Value')
plt.ylabel('Expected Revenue')
plt.title('Bayesian Optimization: Expected Revenue vs. Price')
plt.grid(True)
plt.tight_layout()
plt.show()